# 🎬 Video Q&A Pipeline


This template is an intelligent video analysis system that extracts frames from videos, generates captions using vision transformers, and enables natural language Q&A through Retrieval-Augmented Generation (RAG). Ask questions about video content and get AI-powered answers based on visual understanding.

On [Saturn Cloud](https://saturncloud.io), scale your Video Q&A pipeline from single-GPU prototyping to [distributed multi-GPU inference](https://saturncloud.io/docs/user-guide/how-to/llms/parallel_training/). Process multiple videos simultaneously, parallelize frame captioning across GPUs, accelerate vector embedding generation, and handle high-volume Q&A requests — all in a managed environment with CUDA-optimized OpenCV, PyTorch, and Transformers.


Install dependencies for the template

In [ ]:
!pip install -q --no-deps transformers
!pip install -q --no-deps tokenizers
!pip install -q pillow
!pip install -q --no-deps langchain
!pip install -q --no-deps langchain-community
!pip install -q --no-deps langchain-core
!pip install -q --no-deps langchain-huggingface
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q huggingface-hub

This is to programmatically download a video from a specified URL and save it as a temporary file on the local file system, this is to build the the video Q&A pipeline.

In [ ]:
import requests
import tempfile
import os

# Disable SSL warnings (optional, for insecure URLs)
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def get_video_from_url(url):
    response = requests.get(url, stream=True, verify=False)
    if response.status_code == 200:
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp4", dir="/content")
        temp_file.write(response.content)
        temp_file.flush()
        return temp_file.name
    else:
        raise Exception(f"Failed to download video: {response.status_code}")

# Example video URL
video_url = "https://opencv.org/wp-content/uploads/2025/02/Example-Video.mp4"
video_path = get_video_from_url(video_url)
print("✅ Video downloaded and available at:", video_path)

This extract frames from a video file at specified intervals and saving them as individual image files. It uses the cv2 library (OpenCV) to handle video reading and image writing.

In [ ]:
import cv2

def extract_frames(video_path, output_dir="/content/frames", frame_interval=30):
    """Extract frames from video at specified intervals"""
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_interval == 0:
            frame_path = os.path.join(output_dir, f"frame_{saved_count:04d}.jpg")
            cv2.imwrite(frame_path, frame)
            saved_count += 1

        frame_count += 1

    cap.release()
    print(f"✅ Extracted {saved_count} frames to '{output_dir}/'")
    return saved_count

# Extract frames from the downloaded video
extract_frames(video_path, frame_interval=30)

Performs image captioning using a pre-trained VisionEncoderDecoder model (ViT-GPT2) from Hugging Face Transformers to generate textual descriptions for a series of image frames.

In [ ]:
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from PIL import Image
import torch

# Device config
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load model & processor
print("🔄 Loading ViT-GPT2 image captioning model...")
processor = ViTImageProcessor.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
tokenizer = AutoTokenizer.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
model = VisionEncoderDecoderModel.from_pretrained("nlpconnect/vit-gpt2-image-captioning").to(device)
print("✅ Model loaded.")

# Generate captions for frames
def generate_lightweight_captions(frame_dir="/content/frames", max_frames=11):
    captions = []
    files = sorted(os.listdir(frame_dir))[:max_frames]

    for idx, fname in enumerate(files):
        try:
            path = os.path.join(frame_dir, fname)
            image = Image.open(path).convert("RGB")
            pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
            output_ids = model.generate(pixel_values, max_length=16, num_beams=4)
            caption = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            captions.append(caption)
            print(f"[{idx+1}/{len(files)}] {fname} → {caption}")
        except Exception as e:
            print(f"⚠️ Error processing {fname}: {e}")
            captions.append("Error generating caption")

    return captions

# Run captioning
captions = generate_lightweight_captions("/content/frames", max_frames=11)

This sets up a vector database (ChromaDB) and populates it with text embeddings generated from image captions. It uses the LangChain framework—specifically the langchain_community module—for handling embeddings, vector storage, and document retrieval.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Convert captions into Document objects
docs = [Document(page_content=cap) for cap in captions]

# Load the embedding model
print("🔄 Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Store in ChromaDB
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="/content/chroma_db"
)

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ Vector store created and retriever ready!")
print(f"📊 Indexed {len(captions)} frame captions\n")

This code implements the final step of a video-based question answering (Q&A) pipeline using a lightweight language model from Hugging Face Transformers.

In [ ]:
from transformers import pipeline

# Load a small, efficient model for Q&A (fits in Colab free tier)
print("🔄 Loading lightweight Q&A model...")
qa_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",  # Small model (~300MB)
    device=0 if torch.cuda.is_available() else -1,
    max_length=512
)
print("✅ Q&A model loaded!")

def ask_video_question(query):
    """Ask a question about the video and get an AI-generated answer"""

    # Retrieve relevant frame captions
    docs = retriever.invoke(query)
    context = "\n".join([f"Frame {i+1}: {doc.page_content}" for i, doc in enumerate(docs)])

    # Create prompt
    prompt = f"""Answer the question based on these video frame descriptions:

{context}

Question: {query}
Answer:"""

    # Get response from model
    print(f"\n🔍 Question: {query}")
    print("🤔 Thinking...")

    try:
        response = qa_pipeline(prompt, max_length=256, do_sample=False)[0]['generated_text']
        print(f"✅ Answer: {response}\n")
    except Exception as e:
        # Fallback: just show the relevant captions
        print(f"⚠️ Error generating answer: {e}")
        print(f"📋 Relevant frames found:")
        for i, doc in enumerate(docs, 1):
            print(f"  Frame {i}: {doc.page_content}")
        response = " ".join([doc.page_content for doc in docs])

    return response

# Ask questions about your video
print("="*60)
print("VIDEO Q&A SESSION")
print("="*60)

ask_video_question("What is happening in the video?")
ask_video_question("What objects or people appear in the video?")
ask_video_question("Describe the main activity shown in the video?")

# Optional: Interactive Q&A
print("\n" + "="*60)
print("💡 TIP: You can ask custom questions by calling:")
print('ask_video_question("your question here")')
print("="*60)

This template provides all the tools that is needed to develop advanced video understanding applications—whether you're building video summarization tools, educational video analysis, or AI-driven surveillance systems. 

If you plan to scale up to longer videos or batch-processing video datasets, you can distribute workloads using [Saturn Cloud](https://saturncloud.io).

You can also enhance the performance and specificity of the system by fine-tuning captioning or embedding models on domain-specific video datasets.